In [1]:
# Cell 1
import sys
sys.path.append('../')
import pandas as pd
from src.event_log.ingest import ingest_once
from src.event_log.preprocessor import (
    standardize_columns, filter_lifecycle,
    remove_duplicates, add_trace_features,
    validate_traces, save_processed
)

In [2]:
# Cell 2 — Load từ Parquet (nhanh)
df_raw = ingest_once(
    xes_path    ='../data/raw/BPI_2017.xes',
    parquet_path='../data/raw/BPI_2017_raw.parquet'
)

Parquet đã tồn tại — load từ cache: ../data/raw/BPI_2017_raw.parquet
Load xong: 1202267 event, 19 cột


In [4]:
# Cell 3 — Chuẩn hoá
df = standardize_columns(df_raw)
print(f"Cột sau chuẩn hoá: {df.columns.tolist()}")

Cột sau chuẩn hoá: ['action', 'resource', 'activity', 'event_origin', 'event_id', 'lifecycle', 'timestamp', 'loan_goal', 'application_type', 'case_id', 'requested_amount', 'first_withdrawal_amount', 'number_of_terms', 'accepted', 'monthly_cost', 'selected', 'credit_score', 'offered_amount', 'offer_id']


In [5]:
# Cell 4 — Giữ toàn bộ lifecycle, không lọc
df = filter_lifecycle(df)  # không truyền keep

Giữ toàn bộ lifecycle — không lọc
lifecycle
complete     475306
suspend      215402
schedule     149104
start        128227
resume       127160
ate_abort     85224
withdraw      21844


In [6]:
# Cell 5 — Loại duplicate
df = remove_duplicates(df)

Loại duplicate: 0 event bị loại


In [7]:
# Cell 6 — Phân tích độ dài trace trước khi lọc
from src.event_log.preprocessor import analyze_trace_length
p_low, p_high = analyze_trace_length(df)

Phân phối độ dài trace (31509 case):
  min    = 10
  p5     = 19   ← đề xuất min_events
  p25    = 25
  median = 35
  mean   = 38.2
  p75    = 47
  p95    = 70  ← đề xuất max_events
  max    = 180

Đề xuất: filter_complete_cases(min_events=19, max_events=70)


In [19]:
# Cell 7 — Lọc outlier dựa trên p5/p95
from src.event_log.preprocessor import filter_complete_cases
df = filter_complete_cases(df, min_events=19, max_events=70)

Lọc trace [19, 70] event:
  Giữ: 28512 case
  Loại: 0 case outlier


In [14]:
# Cell 8 — Thêm feature
df = add_trace_features(df)

Đã thêm seq_index, duration_next, is_terminal


In [15]:
# Cell 9 — Kiểm tra chất lượng
df = validate_traces(df)

Thống kê chất lượng trace:
  Tổng case: 28512
  Độ dài trace: min=19, max=70, mean=36.7
  Case có event complete:  28512 (100.0%)
  Case có ate_abort:       28425 (99.7%)
  Case có withdraw:        17122 (60.1%)
  Case có đủ 3 origin:     28512 (100.0%)


In [16]:
# Cell 10 — Lưu
save_processed(df, '../data/processed/event_log_clean.csv')

Đã lưu:
  ../data/processed/event_log_clean.parquet
  ../data/processed/event_log_clean.csv
  (1047482 event, 28512 case, 22 cột)


In [12]:
# Cell 11 — Kiểm tra lại case mẫu
case_id = 'Application_652823628'
case_df = df[df['case_id'] == case_id].sort_values('timestamp')
cols_show = ['seq_index', 'event_origin', 'activity',
             'lifecycle', 'resource', 'duration_next']
print(f"Số event: {len(case_df)}")
print(case_df[cols_show].to_string(index=False))

Số event: 40
 seq_index event_origin                 activity lifecycle resource  duration_next
         0  Application     A_Create Application  complete   User_1              0
         1  Application              A_Submitted  complete   User_1              0
         2     Workflow           W_Handle leads  schedule   User_1             80
         3     Workflow           W_Handle leads  withdraw   User_1              0
         4     Workflow   W_Complete application  schedule   User_1              0
         5  Application                A_Concept  complete   User_1          89566
         6     Workflow   W_Complete application     start  User_17            246
         7     Workflow   W_Complete application   suspend  User_17           2015
         8  Application               A_Accepted  complete  User_52            359
         9        Offer           O_Create Offer  complete  User_52              1
        10        Offer                O_Created  complete  User_52       